# Data Collection 

In [ ]:
import pandas as pd
from dotenv import load_dotenv
import requests
import os
import time

In [6]:
BASE_URL = (
    "https://nces.ed.gov/arcgis/rest/services/"
    "CCD/CCD_Data/MapServer/5"
)

QUERY_URL = f"{BASE_URL}/query"

# ----------------------------------------
# 1. Define variables to collect
# ----------------------------------------

features = [
    "NCESSCH",
    "STUTERATIO",
    "TOTFRL",
    "WH",
    "BL",
    "HI",
    "AS",
    "AM",
    "HP",
    "TR",
    "STABR",
    "ULOCALE",
    "CHARTER_TEXT",
    "VIRTUAL",
    "SCHOOL_TYPE_TEXT",
    "SLEVEL_TEXT",
    "GSLO_NUM",
    "GSHI_NUM",
    "MEMBER",
    "FTE"
]

# ----------------------------------------
# 2. Retrieve all object IDs
# ----------------------------------------

response = requests.get(
    QUERY_URL,
    params={
        "where": "1=1",
        "returnIdsOnly": "true",
        "f": "json"
    },
    timeout=120
)

response.raise_for_status()

data = response.json()

if "error" in data:
    raise RuntimeError(data["error"])

object_ids = data["objectIds"]

print("Total schools:", len(object_ids))

# ----------------------------------------
# 3. Retrieve records in batches
# ----------------------------------------

batch_size = 500

all_records = []

for i in range(0, len(object_ids), batch_size):

    batch_ids = object_ids[i:i + batch_size]

    params = {
        "objectIds": ",".join(map(str, batch_ids)),
        "outFields": ",".join(features),
        "returnGeometry": "false",
        "f": "json"
    }

    response = requests.get(
        QUERY_URL,
        params=params,
        timeout=120
    )

    response.raise_for_status()

    result = response.json()

    if "error" in result:
        raise RuntimeError(result["error"])

    records = [
        feature["attributes"]
        for feature in result["features"]
    ]

    all_records.extend(records)

    print(
        f"Retrieved {len(all_records)} / "
        f"{len(object_ids)} records"
    )

    time.sleep(0.2)

# ----------------------------------------
# 4. Convert to DataFrame
# ----------------------------------------

df = pd.DataFrame(all_records)

print("Dataset shape:", df.shape)

display(df.head())

# # ----------------------------------------
# # 5. Save raw dataset
# # ----------------------------------------

# df.to_csv(
#     "ccd_2025_raw.csv",
#     index=False
# )

# print("Saved ccd_2025_raw.csv")

Total schools: 101110
Retrieved 500 / 101110 records
Retrieved 1000 / 101110 records
Retrieved 1500 / 101110 records
Retrieved 2000 / 101110 records
Retrieved 2500 / 101110 records
Retrieved 3000 / 101110 records
Retrieved 3500 / 101110 records
Retrieved 4000 / 101110 records
Retrieved 4500 / 101110 records
Retrieved 5000 / 101110 records
Retrieved 5500 / 101110 records
Retrieved 6000 / 101110 records
Retrieved 6500 / 101110 records
Retrieved 7000 / 101110 records
Retrieved 7500 / 101110 records
Retrieved 8000 / 101110 records
Retrieved 8500 / 101110 records
Retrieved 9000 / 101110 records
Retrieved 9500 / 101110 records
Retrieved 10000 / 101110 records
Retrieved 10500 / 101110 records
Retrieved 11000 / 101110 records
Retrieved 11500 / 101110 records
Retrieved 12000 / 101110 records
Retrieved 12500 / 101110 records
Retrieved 13000 / 101110 records
Retrieved 13500 / 101110 records
Retrieved 14000 / 101110 records
Retrieved 14500 / 101110 records
Retrieved 15000 / 101110 records
Retrieve

,NCESSCH,STUTERATIO,TOTFRL,WH,BL,HI,AS,AM,HP,TR,STABR,ULOCALE,CHARTER_TEXT,VIRTUAL,SCHOOL_TYPE_TEXT,SLEVEL_TEXT,GSLO_NUM,GSHI_NUM,MEMBER,FTE
0,010000500870,19.74,562,262,45,518,3,4,0,17,AL,32-Town: Distant,No,Not Virtual,Regular School,Middle,7,8.0,849,43.0
1,010000500871,18.71,798,419,56,714,7,5,1,42,AL,32-Town: Distant,No,Not Virtual,Regular School,High,9,12.0,1244,66.5
2,010000500879,20.07,570,267,46,496,6,2,0,26,AL,32-Town: Distant,No,Not Virtual,Regular School,Middle,5,6.0,843,42.0
3,010000500889,17.86,628,259,56,567,2,10,1,16,AL,32-Town: Distant,No,Not Virtual,Regular School,Elementary,3,4.0,911,51.0
4,010000501616,18.26,304,193,54,304,2,1,0,12,AL,32-Town: Distant,No,Not Virtual,Regular School,Elementary,-4,0.0,566,31.0


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101110 entries, 0 to 101109
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   NCESSCH           101110 non-null  str    
 1   STUTERATIO        101110 non-null  float64
 2   TOTFRL            101110 non-null  int64  
 3   WH                101110 non-null  int64  
 4   BL                101110 non-null  int64  
 5   HI                101110 non-null  int64  
 6   AS                101110 non-null  int64  
 7   AM                101110 non-null  int64  
 8   HP                101110 non-null  int64  
 9   TR                101110 non-null  int64  
 10  STABR             101110 non-null  str    
 11  ULOCALE           101110 non-null  str    
 12  CHARTER_TEXT      101110 non-null  str    
 13  VIRTUAL           101110 non-null  str    
 14  SCHOOL_TYPE_TEXT  101110 non-null  str    
 15  SLEVEL_TEXT       101110 non-null  str    
 16  GSLO_NUM          101110 non-nu